In [ ]:
pip install --upgrade pip

In [ ]:
pip install pymupdf openai tqdm

In [ ]:
pip install pdf2image openai python-dotenv tqdm


In [ ]:
pip install PyPDF2 

In [ ]:
pip install --upgrade llama-index

In [ ]:
pip install torch sentence-transformers

In [ ]:
pip install hf_xet

In [ ]:
import base64
import os
from tqdm import tqdm
from openai import AzureOpenAI
from pdf2image import convert_from_path
from dotenv import load_dotenv
from PyPDF2 import PdfReader
import gc

In [ ]:
# This reads the .env file and sets the environment variables
load_dotenv()

In [ ]:
# Connect to Azure OpenAI
client = AzureOpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    api_version=os.environ["OPENAI_API_VERSION"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"]
)


In [ ]:
# Function to describe an image
def describe_page_image(pil_image):
    # Convert PIL image to base64
    import io
    buffered = io.BytesIO()
    pil_image.save(buffered, format="PNG")
    img_base64 = base64.b64encode(buffered.getvalue()).decode()

    data_url = f"data:image/png;base64,{img_base64}"

    prompt = """
You are a professional technical writer.
Analyze the following page from a PDF document.
Generate a detailed Markdown report that includes:
- Extracted text
- Descriptions of any plots, charts, graphs, or tables
- Any geographic references or maps
Use Markdown syntax cleanly and professionally.
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": data_url}}
                ]
            }
        ],
        temperature=0.2
    )
    return response.choices[0].message.content

In [ ]:
# Function to process an entire PDF
def process_pdf_to_markdown(input_pdf_path, output_md_path, dpi=300, pages_per_chunk=60):
    print(f"🔄 Rendering PDF: {input_pdf_path}")

    # Get the total number of pages
    reader = PdfReader(input_pdf_path)
    total_pages = len(reader.pages)

    all_markdown = []

    page_num = 0
    chunk_num = 1

    while page_num < total_pages:
        end_page = min(page_num + pages_per_chunk, total_pages)
        print(f"📚 Processing chunk {chunk_num}: pages {page_num+1} to {end_page}")

        for p in tqdm(range(page_num, end_page), desc=f"Chunk {chunk_num}"):
            pil_image = convert_from_path(
                input_pdf_path,
                dpi=dpi,
                first_page=p + 1,
                last_page=p + 1
            )[0]  # Only one page returned

            page_md = f"\n\n## Page {p + 1}\n\n"

            try:
                page_description = describe_page_image(pil_image)
                page_md += page_description
            except Exception as e:
                page_md += f"_Error processing page {p + 1}: {str(e)}_"

            all_markdown.append(page_md)

            # Immediate memory cleanup
            del pil_image
            gc.collect()

        page_num = end_page
        chunk_num += 1

    # Save final Markdown
    with open(output_md_path, "w", encoding="utf-8") as f:
        f.write("\n\n".join(all_markdown))

In [ ]:
# This cell conversts one single pdf to markdown
input_pdf = "./data/source_files/Meta-2021-Sustainability-Report.pdf"  
output_md = "./data/md_files/Meta-2021-Sustainability-Report.md"

# input_pdf = "./data/source_files/Netflix_2022-ESG-Report-FINAL.pdf"  
# output_md = "./data/md_files/Netflix_2022-ESG-Report-FINAL.md"

process_pdf_to_markdown(input_pdf, output_md)

In [ ]:
# Function to run the process for all the files in the folder
def list_files_without_extension(base_path):
    source_path = base_path+"/source_files/"
    md_path = base_path+"/md_files/"

    file_list = os.listdir(source_path)
    
    files_processed = []
    for file_name in file_list:
        if os.path.isfile(os.path.join(source_path, file_name)):
            name, ext = os.path.splitext(file_name)

            input_pdf = source_path + file_name
            output_md = md_path + name + ".md"
            process_pdf_to_markdown(input_pdf, output_md)

            files_processed.append(name)
    return files_processed

In [ ]:
base_path = "./data"
list_files_without_extension(base_path)

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding
from llama_index.core import Settings
from llama_index.core import VectorStoreIndex
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.query_pipeline import QueryPipeline

import ipywidgets as widgets
from IPython.display import display, clear_output
import time

In [ ]:
# Load documents
documents = SimpleDirectoryReader(input_dir="./data/md_files/").load_data(
    show_progress=True
)

# Build a better parser
text_splitter = SentenceSplitter(
    chunk_size=256,   # How big you want each embedded unit
    chunk_overlap=50,  # Small overlap between chunks to preserve context
    paragraph_separator= "\n\n"  # Keeps paragraphs intact better
)

# Parse documents into nice small nodes
nodes = text_splitter.get_nodes_from_documents(documents)

# Then embed
embed_model = AzureOpenAIEmbedding(
    model="text-embedding-3-small",
    api_key=os.environ["OPENAI_API_KEY"],
    api_version=os.environ["OPENAI_API_VERSION"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"]
)
Settings.embed_model = embed_model

# Now build the index over the cleaned nodes
index = VectorStoreIndex(nodes, show_progress=True)

In [ ]:
# LLM Setup
llm = AzureOpenAI(
    deployment_name="gpt-4o",
    model="gpt-4o",
    temperature=0.0,
    api_key=os.environ["OPENAI_API_KEY"],
    api_version=os.environ["OPENAI_API_VERSION"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"]
)
Settings.llm = llm

# Build a custom retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=8,    # number of results to fetch
    vector_similarity_top_k=8,  # top k for vector search
    hybrid=True            # Enable hybrid retriever (vector + keyword)
)

# Load a fast, high-quality reranker model from Hugging Face
reranker = SentenceTransformerRerank(
    top_n=3,  # how many reranked chunks to keep
    # model="cross-encoder/ms-marco-MiniLM-L-6-v2"  # small and powerful
    model="cross-encoder/ms-marco-electra-base"    # More powerful but slower
)

# Build the final query engine
query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever,
    llm=llm,
    node_postprocessors=[reranker]  # << reranker goes here
)

# Query Engine Setup
query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever,
    llm=llm   # <-- explicitly pass the llm
)

In [ ]:
# Function to get a response
def get_response(user_input):
    return query_engine.query(user_input).response

# Interactive chatbot function
def chatbot():
    output = widgets.Output()
    messages = []  # Save conversation history

    scrollable_chat = widgets.Box(
        [output],
        layout=widgets.Layout(
            width="100%",
            height="400px",
            overflow="auto",
            border="1px solid #ccc",
            padding="5px",
            display="flex",
            flex_flow="column"
        )
    )

    text_box = widgets.Text(
        placeholder="Type your message here",
        description="You:",
        style={'description_width': 'initial'},
        layout=widgets.Layout(width="70%")
    )
    
    submit_button = widgets.Button(
        description="Send",
        button_style="primary",
        layout=widgets.Layout(width="15%")
    )
    
    clear_button = widgets.Button(
        description="Clear Chat",
        button_style="warning",
        layout=widgets.Layout(width="15%")
    )

    button_box = widgets.HBox([text_box, submit_button, clear_button])
    chat_container = widgets.VBox([scrollable_chat, button_box])
    
    def on_submit(_):
        user_message = text_box.value.strip()
        if not user_message:
            return
        
        with output:
            # Display user message
            display(widgets.HTML(
                f"<div style='color: blue; font-weight: bold;'>You:</div> {user_message}"
            ))
        
        try:
            bot_response = get_response(user_message)
            messages.append((user_message, bot_response))  # Save conversation
            
            with output:
                # Streaming simulation (letter-by-letter reveal)
                bot_output = widgets.HTML(value=f"<div style='color: green; font-weight: bold;'>Bot:</div> ")
                display(bot_output)
                full_response = ""
                for char in bot_response:
                    full_response += char
                    bot_output.value = f"<div style='color: green; font-weight: bold;'>Bot:</div> {full_response}"
                    time.sleep(0.005)  # tiny delay for effect

        except Exception as e:
            with output:
                display(widgets.HTML(
                    f"<div style='color: red;'>⚠️ Error: {str(e)}</div>"
                ))
        
        text_box.value = ""
        
    def on_clear(_):
        output.clear_output()
        messages.clear()
    
    # Attach events
    submit_button.on_click(on_submit)
    clear_button.on_click(on_clear)

    display(chat_container)

# Run the chatbot
chatbot()

# How many people in India did Microsoft provide water access to?
# What is the objective of the Carbon Call?

## Let's evaluate it

In [ ]:
from llama_index.core.llama_dataset import LabelledRagDataset
from IPython.display import display, Markdown, clear_output
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

In [ ]:
rag_dataset = LabelledRagDataset.from_json("./data/rag_dataset.json")

In [ ]:
# Load some questions as example

num_samples = 5
samples = rag_dataset.to_pandas().loc[:num_samples-1, 'query'].values
display(Markdown("\n".join([ "* " + sample for sample in samples])))

In [ ]:
rag_dataset.to_pandas().head()

Now, let's calculate the responses for the dataset:

In [ ]:
predictions = rag_dataset.make_predictions_with(
    predictor = query_engine,
    show_progress = True
)

In [ ]:
list_of_samples = []

for idx in range(len(rag_dataset.examples)):
    list_of_samples.append(
        SingleTurnSample (
            user_input = rag_dataset.examples[idx].query,
            reference = rag_dataset.examples[idx].reference_answer,
            response = predictions.predictions[idx].response,
            retrieved_contexts = predictions.predictions[idx].contexts
        )
    )

ragas_evaluation_dataset = EvaluationDataset(list_of_samples)
ragas_evaluation_dataset.to_pandas().head()

### Initialize the LLM and Embedding Models

In [ ]:
# To calculate the Generative AI quality metrics
from ragas.llms import LlamaIndexLLMWrapper
from ragas.embeddings import LlamaIndexEmbeddingsWrapper
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from ragas.evaluation import evaluate
from ragas.run_config import RunConfig

from ragas.metrics import (
    Faithfulness,
    ContextPrecision,
    ContextRecall
)

import pandas as pd

In [ ]:
evaluator_llm = LlamaIndexLLMWrapper(llm)
evaluator_embeddings = LlamaIndexEmbeddingsWrapper(embed_model)

In [ ]:
metrics = [
    Faithfulness(llm=evaluator_llm),
    ContextPrecision(llm=evaluator_llm),
    ContextRecall(llm=evaluator_llm)
]
ragas_evaluation_result = evaluate(
    dataset=ragas_evaluation_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=RunConfig(timeout=1800, max_wait=180, max_retries=20),
    show_progress=True,
    batch_size=5
)

### let's compare

In [ ]:
df_ragas_result = ragas_evaluation_result.to_pandas()

In [ ]:
df_old_dataset = pd.read_json('./test-dataset.json', orient='records')

In [ ]:
# Compare the results

values_df1 = []
values_df2 = []

for i in range(4,7):
    values_df1.append(df_ragas_result.iloc[:, i].mean())
    values_df2.append(df_old_dataset.iloc[:, i].mean())
    
# Create a markdown table
variable_names = df_old_dataset.columns[4:]

df_results = pd.DataFrame({
    "Variable": variable_names,
    "New results": values_df1,
    "Old results": values_df2
})

# Display it nicely
display(df_results)


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
 # Plot the actual vs estimated values for the selected metric

def plot_a_metric(metric, df1, df2):

    plt.figure(figsize=(20,10))
    plt.plot(df1[metric], label='New Value', marker='o', markersize = 10)
    plt.plot(df2[metric], label='Old Value', marker='s')
    
    
    plt.xlabel('Index')
    plt.ylabel('Value')
    plt.title(f'{metric}: New vs Old')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# metric = "context_precision"

for metric in df_old_dataset.columns[4:]:
    plot_a_metric(metric, df_ragas_result, df_old_dataset)